# PyTorch 常用工具速览

这份 Notebook 是一张 PyTorch 工具地图。目标不是记住所有 API，而是先知道：PyTorch 能做什么、常用工具属于哪一类、以后遇到问题该去哪里找。

> 建议：按顺序运行一遍，观察输出；暂时不理解的细节可以先跳过。

## 0. 导入与版本

PyTorch 通常简称为 `torch`。张量（Tensor）是最核心的数据结构，可以把它理解为支持 GPU 和自动求导的多维数组。

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print("PyTorch version:", torch.__version__)

PyTorch version: 2.8.0.dev20250525


## 1. 创建张量

最常用的创建方式：

| 工具 | 用途 |
| --- | --- |
| `torch.tensor(data)` | 从 Python 列表或现有数据创建 |
| `torch.zeros(shape)` | 全 0 张量 |
| `torch.ones(shape)` | 全 1 张量 |
| `torch.full(shape, value)` | 填充指定值 |
| `torch.empty(shape)` | 只分配内存，不初始化数值 |
| `torch.arange(start, end, step)` | 等间隔整数序列，不包含终点 |
| `torch.linspace(start, end, steps)` | 在区间中取固定数量的点 |
| `torch.eye(n)` | 单位矩阵 |
| `torch.zeros_like(x)` 等 | 创建与 `x` 形状、类型、设备相同的张量 |

In [22]:
a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
print("tensor:\n", a)
print("zeros:\n", torch.zeros(2, 3))
print("ones:\n", torch.ones(2, 3))
print("full:\n", torch.full((2, 3), 7))
print("arange:", torch.arange(0, 10, 2))
print("linspace:", torch.linspace(0, 1, 5))
print("eye:\n", torch.eye(3))
print("zeros_like:\n", torch.zeros_like(a))

tensor:
 tensor([[1., 2.],
        [3., 4.]])
zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
full:
 tensor([[7, 7, 7],
        [7, 7, 7]])
arange: tensor([0, 2, 4, 6, 8])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
eye:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
zeros_like:
 tensor([[0., 0.],
        [0., 0.]])


## 2. 随机数与随机分布

随机张量常用于初始化参数、生成模拟数据和打乱样本。

| 工具 | 分布或用途 |
| --- | --- |
| `torch.rand(shape)` | `[0, 1)` 均匀分布 |
| `torch.randn(shape)` | 标准正态分布，均值 0、标准差 1 |
| `torch.normal(mean, std, size)` | 自定义均值和标准差的正态分布 |
| `torch.randint(low, high, shape)` | 指定区间的随机整数 |
| `torch.randperm(n)` | `0` 到 `n-1` 的随机排列 |
| `torch.bernoulli(p)` | 按概率生成 0 或 1 |
| `torch.multinomial(weights, n)` | 按权重抽样 |
| `torch.manual_seed(seed)` | 固定随机种子，便于复现实验 |

In [4]:
torch.manual_seed(42)
print("rand:", torch.rand(4))
print("randn:", torch.randn(4))
print("normal:", torch.normal(mean=10.0, std=2.0, size=(4,)))
print("randint:", torch.randint(0, 10, (4,)))
print("randperm:", torch.randperm(6))
print("bernoulli:", torch.bernoulli(torch.tensor([0.2, 0.5, 0.8])))

rand: tensor([0.8823, 0.9150, 0.3829, 0.9593])
randn: tensor([ 0.2345,  0.2303, -1.1229, -0.1863])
normal: tensor([14.4164,  8.7240, 10.9233, 10.5347])
randint: tensor([9, 6, 3, 1])
randperm: tensor([5, 4, 3, 2, 0, 1])
bernoulli: tensor([1., 0., 1.])


### 分布对象

需要更完整的概率分布功能时，使用 `torch.distributions`。常见对象包括 `Normal`、`Uniform`、`Bernoulli`、`Categorical`。它们可以采样，也可以计算概率或对数概率。

In [5]:
normal_dist = torch.distributions.Normal(loc=0.0, scale=1.0)
samples = normal_dist.sample((5,))
print("samples:", samples)
print("log probabilities:", normal_dist.log_prob(samples))

samples: tensor([ 1.3221,  0.8172, -0.7658, -0.7506,  1.3525])
log probabilities: tensor([-1.7930, -1.2528, -1.2122, -1.2007, -1.8336])


## 3. 查看张量信息与类型转换

遇到错误时，优先查看 `shape`、`dtype` 和 `device`。

| 工具 | 用途 |
| --- | --- |
| `x.shape` / `x.size()` | 各维度大小 |
| `x.ndim` | 维度数量 |
| `x.numel()` | 元素总数 |
| `x.dtype` | 数据类型 |
| `x.device` | 所在设备 |
| `x.float()` / `x.long()` / `x.to(dtype)` | 转换数据类型 |
| `x.item()` | 单元素张量转为 Python 标量 |
| `x.tolist()` | 张量转为 Python 列表 |
| `torch.from_numpy(array)` / `x.numpy()` | 与 NumPy 互转（CPU 张量） |

In [6]:
x = torch.arange(12).reshape(3, 4)
print("shape:", x.shape, "ndim:", x.ndim, "numel:", x.numel())
print("dtype:", x.dtype, "device:", x.device)
print("as float:", x.float().dtype)
print("scalar item:", x[0, 0].item())

shape: torch.Size([3, 4]) ndim: 2 numel: 12
dtype: torch.int64 device: cpu
as float: torch.float32
scalar item: 0


## 4. 改变形状和维度

| 工具 | 用途 |
| --- | --- |
| `reshape` / `view` | 改变形状；`reshape` 通常更稳妥 |
| `flatten` | 展平多个维度 |
| `squeeze` | 删除大小为 1 的维度 |
| `unsqueeze` | 插入大小为 1 的维度 |
| `transpose` | 交换两个维度 |
| `permute` | 任意重排多个维度 |
| `expand` | 广播式扩展，不复制数据 |
| `repeat` | 重复数据，会真正复制 |

In [7]:
x = torch.arange(12).reshape(3, 4)
print("reshape:", x.reshape(2, 6).shape)
print("flatten:", x.flatten().shape)
print("unsqueeze:", x.unsqueeze(0).shape)
print("transpose:", x.transpose(0, 1).shape)
image_batch = torch.rand(8, 3, 32, 32)  # N, C, H, W
print("permute NCHW -> NHWC:", image_batch.permute(0, 2, 3, 1).shape)

reshape: torch.Size([2, 6])
flatten: torch.Size([12])
unsqueeze: torch.Size([1, 3, 4])
transpose: torch.Size([4, 3])
permute NCHW -> NHWC: torch.Size([8, 32, 32, 3])


## 5. 索引、切片、筛选与组合

张量支持类似 NumPy 的索引。常用组合工具有：

| 工具 | 用途 |
| --- | --- |
| `x[...]` | 索引和切片 |
| 布尔索引 | 按条件筛选 |
| `torch.where` | 按条件选择两个值 |
| `torch.cat` | 沿已有维度拼接 |
| `torch.stack` | 创建新维度后堆叠 |
| `torch.split` / `torch.chunk` | 拆分张量 |
| `gather` / `scatter` | 按索引收集或写入，进阶模型中常见 |

In [8]:
x = torch.arange(12).reshape(3, 4)
print("first row:", x[0])
print("last two columns:\n", x[:, -2:])
print("values > 5:", x[x > 5])
print("where:\n", torch.where(x > 5, x, torch.tensor(-1)))
print("cat shape:", torch.cat([x, x], dim=0).shape)
print("stack shape:", torch.stack([x, x], dim=0).shape)

first row: tensor([0, 1, 2, 3])
last two columns:
 tensor([[ 2,  3],
        [ 6,  7],
        [10, 11]])
values > 5: tensor([ 6,  7,  8,  9, 10, 11])
where:
 tensor([[-1, -1, -1, -1],
        [-1, -1,  6,  7],
        [ 8,  9, 10, 11]])
cat shape: torch.Size([6, 4])
stack shape: torch.Size([2, 3, 4])


## 6. 数学运算与广播

### 逐元素运算

`+`、`-`、`*`、`/`、`**` 都是逐元素运算。常见函数还有 `abs`、`sqrt`、`exp`、`log`、`sin`、`cos`、`clamp`。形状不同时，PyTorch 会尝试广播（从末尾维度对齐）。

### 聚合运算

`sum`、`mean`、`min`、`max`、`argmin`、`argmax`、`std`、`var` 会把许多数值汇总为更少的数值。通过 `dim` 指定沿哪个维度计算，`keepdim=True` 可以保留该维度。

In [9]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
bias = torch.tensor([10.0, 20.0, 30.0])
print("broadcast add:\n", x + bias)
print("sqrt:\n", torch.sqrt(x))
print("clamp:\n", torch.clamp(x, min=2.0, max=5.0))
print("total mean:", x.mean())
print("column sums:", x.sum(dim=0))
print("row argmax:", x.argmax(dim=1))

broadcast add:
 tensor([[11., 22., 33.],
        [14., 25., 36.]])
sqrt:
 tensor([[1.0000, 1.4142, 1.7321],
        [2.0000, 2.2361, 2.4495]])
clamp:
 tensor([[2., 2., 3.],
        [4., 5., 5.]])
total mean: tensor(3.5000)
column sums: tensor([5., 7., 9.])
row argmax: tensor([2, 2])


## 7. 线性代数

| 工具 | 用途 |
| --- | --- |
| `@` / `torch.matmul` | 矩阵乘法，支持批次维度 |
| `torch.mm` | 两个二维矩阵相乘 |
| `torch.bmm` | 批量矩阵乘法 |
| `torch.dot` | 两个一维向量点积 |
| `torch.einsum` | 用下标表达复杂张量运算 |
| `torch.linalg.norm` | 范数 |
| `torch.linalg.solve` | 解线性方程组 |
| `torch.linalg.inv` / `svd` / `eig` | 逆、奇异值分解、特征分解 |

In [10]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[2.0, 0.0], [1.0, 2.0]])
print("matrix multiply:\n", a @ b)
print("elementwise multiply:\n", a * b)
print("norm:", torch.linalg.norm(a))
print("solve Ax=b:", torch.linalg.solve(a, torch.tensor([1.0, 0.0])))

matrix multiply:
 tensor([[ 4.,  4.],
        [10.,  8.]])
elementwise multiply:
 tensor([[2., 0.],
        [3., 8.]])
norm: tensor(5.4772)
solve Ax=b: tensor([-2.0000,  1.5000])


## 8. 比较、排序与常用统计

| 工具 | 用途 |
| --- | --- |
| `torch.eq` 或 `==` | 逐元素比较 |
| `torch.isclose` / `torch.allclose` | 浮点数近似比较 |
| `torch.sort` / `torch.argsort` | 排序并返回值或索引 |
| `torch.topk` | 取最大的 k 个值和索引 |
| `torch.unique` | 去重 |
| `torch.bincount` | 统计非负整数出现次数 |
| `torch.any` / `torch.all` | 判断是否任意或全部满足条件 |
| `torch.isnan` / `torch.isinf` / `torch.isfinite` | 检查异常数值 |

In [11]:
scores = torch.tensor([0.2, 0.9, 0.4, 0.7])
print("top-2:", torch.topk(scores, k=2))
print("sorted:", torch.sort(scores))
labels = torch.tensor([0, 1, 1, 2, 2, 2])
print("counts:", torch.bincount(labels))
print("all finite:", torch.isfinite(scores).all().item())

top-2: torch.return_types.topk(
values=tensor([0.9000, 0.7000]),
indices=tensor([1, 3]))
sorted: torch.return_types.sort(
values=tensor([0.2000, 0.4000, 0.7000, 0.9000]),
indices=tensor([0, 2, 3, 1]))
counts: tensor([1, 2, 3])
all finite: True


## 9. CPU、CUDA 与 MPS 设备

`torch.device` 表示计算设备。NVIDIA GPU 使用 `cuda`，Apple 芯片 GPU 使用 `mps`，其他情况使用 `cpu`。模型和输入必须在同一设备。

常用工具：`torch.cuda.is_available()`、`torch.backends.mps.is_available()`、`x.to(device)`、`model.to(device)`、`x.cpu()`。

In [12]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

x = torch.rand(2, 3).to(device)
print("selected device:", device)
print("tensor device:", x.device)

selected device: mps
tensor device: mps:0


## 10. 自动微分（Autograd）

PyTorch 可以记录张量运算并自动计算梯度。

| 工具 | 用途 |
| --- | --- |
| `requires_grad=True` | 要求追踪该张量的运算 |
| `loss.backward()` | 从标量损失反向计算梯度 |
| `x.grad` | 查看叶子张量累积的梯度 |
| `x.detach()` | 得到与计算图分离的张量 |
| `torch.no_grad()` | 临时关闭梯度记录 |
| `torch.inference_mode()` | 推理时更彻底地关闭 autograd 开销 |
| `torch.autograd.grad` | 直接计算指定输出对输入的梯度 |

In [13]:
w = torch.tensor(2.0, requires_grad=True)
loss = (w * 3 - 10) ** 2
loss.backward()
print("loss:", loss.item(), "gradient:", w.grad.item())

with torch.inference_mode():
    prediction = w * 3
print("inference prediction:", prediction.item())

loss: 16.0 gradient: -24.0
inference prediction: 6.0


## 11. 构建神经网络：`torch.nn`

模型通常继承 `nn.Module`，或者用 `nn.Sequential` 快速组合层。

常见层：

- 全连接：`nn.Linear`
- 卷积：`nn.Conv1d`、`nn.Conv2d`、`nn.Conv3d`
- 循环网络：`nn.RNN`、`nn.GRU`、`nn.LSTM`
- 注意力：`nn.MultiheadAttention`、`nn.Transformer`
- 归一化：`nn.BatchNorm2d`、`nn.LayerNorm`
- 正则化：`nn.Dropout`
- 激活：`nn.ReLU`、`nn.GELU`、`nn.Sigmoid`、`nn.Tanh`
- 池化：`nn.MaxPool2d`、`nn.AdaptiveAvgPool2d`
- 词嵌入：`nn.Embedding`

In [14]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(8, 3),
)
inputs = torch.randn(5, 4)
logits = model(inputs)
print(model)
print("input shape:", inputs.shape, "output shape:", logits.shape)
print("parameter count:", sum(p.numel() for p in model.parameters()))

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=8, out_features=3, bias=True)
)
input shape: torch.Size([5, 4]) output shape: torch.Size([5, 3])
parameter count: 67


## 12. 激活函数与函数式 API

许多操作同时存在模块形式和函数形式。例如，层可以写成 `nn.ReLU()`，临时计算可以写成 `torch.relu(x)` 或 `torch.nn.functional.relu(x)`。

常见函数：`relu`、`gelu`、`sigmoid`、`tanh`、`softmax`、`log_softmax`、`one_hot`、`normalize`、`pad`。函数式 API 通常简称 `F`：`import torch.nn.functional as F`。

In [15]:
import torch.nn.functional as F

logits = torch.tensor([[1.0, 2.0, 0.5]])
print("relu:", F.relu(logits))
print("softmax probabilities:", F.softmax(logits, dim=1))
print("one hot:", F.one_hot(torch.tensor([0, 2, 1]), num_classes=3))

relu: tensor([[1.0000, 2.0000, 0.5000]])
softmax probabilities: tensor([[0.2312, 0.6285, 0.1402]])
one hot: tensor([[1, 0, 0],
        [0, 0, 1],
        [0, 1, 0]])


## 13. 损失函数

损失函数衡量预测与目标的差距。常见选择：

| 任务 | 常用损失 |
| --- | --- |
| 回归 | `nn.MSELoss`、`nn.L1Loss`、`nn.SmoothL1Loss` |
| 多分类 | `nn.CrossEntropyLoss` |
| 二分类或多标签分类 | `nn.BCEWithLogitsLoss` |
| 类别分布比较 | `nn.KLDivLoss` |
| 相似度学习 | `nn.CosineEmbeddingLoss`、`nn.TripletMarginLoss` |

`CrossEntropyLoss` 直接接收原始 logits，不要提前做 softmax；`BCEWithLogitsLoss` 也不要提前做 sigmoid。

In [16]:
logits = torch.tensor([[2.0, 0.5, -1.0], [0.2, 1.5, 0.3]])
targets = torch.tensor([0, 1])
classification_loss = nn.CrossEntropyLoss()(logits, targets)
print("cross entropy:", classification_loss.item())

predictions = torch.tensor([2.5, 3.5])
values = torch.tensor([3.0, 3.0])
print("MSE:", nn.MSELoss()(predictions, values).item())

cross entropy: 0.347378671169281
MSE: 0.25


## 14. 优化器与学习率调度

优化器根据梯度更新模型参数。最常见的是 `torch.optim.SGD`、`Adam`、`AdamW`、`RMSprop`。

基本顺序是：

1. `optimizer.zero_grad()` 清空旧梯度。
2. 前向计算预测和损失。
3. `loss.backward()` 计算梯度。
4. `optimizer.step()` 更新参数。

学习率调度器在训练过程中调整学习率，常见的有 `StepLR`、`CosineAnnealingLR`、`ReduceLROnPlateau`、`OneCycleLR`。

In [17]:
tiny_model = nn.Linear(1, 1)
optimizer = torch.optim.Adam(tiny_model.parameters(), lr=0.01)
x_train = torch.tensor([[1.0], [2.0], [3.0]])
y_train = torch.tensor([[2.0], [4.0], [6.0]])

optimizer.zero_grad()
loss = nn.MSELoss()(tiny_model(x_train), y_train)
loss.backward()
optimizer.step()
print("one training step, loss:", loss.item())

one training step, loss: 17.774255752563477


## 15. 数据集与 DataLoader

`Dataset` 定义如何取得单个样本，`DataLoader` 负责分批、打乱和并行读取。

常用工具：

- `TensorDataset`：直接把多个张量包装成数据集。
- `DataLoader`：按 batch 迭代数据。
- `random_split`：随机划分训练集、验证集。
- `Subset`：取数据集的子集。
- `ConcatDataset`：连接多个数据集。
- `WeightedRandomSampler`：按权重采样。
- 自定义 `Dataset`：实现 `__len__` 和 `__getitem__`。

In [18]:
features = torch.randn(20, 4)
labels = torch.randint(0, 3, (20,))
dataset = TensorDataset(features, labels)
loader = DataLoader(dataset, batch_size=6, shuffle=True)

batch_features, batch_labels = next(iter(loader))
print("dataset size:", len(dataset))
print("batch shapes:", batch_features.shape, batch_labels.shape)

dataset size: 20
batch shapes: torch.Size([6, 4]) torch.Size([6])


## 16. 训练模式、评估模式和预测

`model.train()` 启用训练行为，`model.eval()` 启用评估行为。它们主要影响 Dropout 和 BatchNorm。评估时通常再搭配 `torch.inference_mode()`。

In [19]:
model.eval()
with torch.inference_mode():
    logits = model(torch.randn(2, 4))
    probabilities = torch.softmax(logits, dim=1)
    predicted_classes = logits.argmax(dim=1)

print("probabilities:\n", probabilities)
print("predicted classes:", predicted_classes)

probabilities:
 tensor([[0.4078, 0.2271, 0.3651],
        [0.2555, 0.3637, 0.3808]])
predicted classes: tensor([0, 2])


## 17. 保存与加载

常用工具是 `torch.save` 和 `torch.load`。通常保存 `model.state_dict()`，而不是直接保存整个模型对象。恢复训练时还要保存优化器状态、epoch 和指标。

In [20]:
import io

buffer = io.BytesIO()
torch.save({"model": model.state_dict()}, buffer)
buffer.seek(0)
checkpoint = torch.load(buffer, weights_only=True)
model.load_state_dict(checkpoint["model"])
print("state dict keys:", list(checkpoint["model"].keys()))

state dict keys: ['0.weight', '0.bias', '3.weight', '3.bias']


## 18. 初始化、梯度处理与调试工具

| 工具 | 用途 |
| --- | --- |
| `torch.nn.init` | Xavier、Kaiming 等参数初始化 |
| `torch.nn.utils.clip_grad_norm_` | 裁剪梯度范数，常用于 RNN/Transformer |
| `model.parameters()` | 遍历可训练参数 |
| `model.named_parameters()` | 同时查看参数名和参数 |
| `model.state_dict()` | 查看可保存的参数和缓冲区 |
| `torch.autograd.set_detect_anomaly(True)` | 定位异常反向传播，速度较慢 |
| `torch.testing.assert_close` | 测试两个张量是否足够接近 |
| `torch.profiler` | 分析 CPU/GPU 时间和内存 |

In [21]:
linear = nn.Linear(4, 2)
nn.init.xavier_uniform_(linear.weight)
for name, parameter in linear.named_parameters():
    print(name, parameter.shape, "requires_grad=", parameter.requires_grad)

torch.testing.assert_close(torch.tensor([1.0]), torch.tensor([1.0 + 1e-6]))
print("tensor comparison passed")

weight torch.Size([2, 4]) requires_grad= True
bias torch.Size([2]) requires_grad= True
tensor comparison passed


## 19. 编译、混合精度和分布式训练（先认识名字）

这些工具在模型较大或训练较慢时再深入：

- `torch.compile(model)`：编译并优化模型执行。
- `torch.amp.autocast`：使用混合精度加速计算、节省显存。
- `torch.amp.GradScaler`：CUDA 混合精度训练时缩放梯度。
- `torch.distributed`：多进程、多 GPU 分布式通信。
- `DistributedDataParallel`：常用的多 GPU 训练封装。
- `torch.export` / `torch.onnx`：导出模型供其他运行时使用。

## 20. 一次训练的完整流程

把前面的工具串起来，典型训练流程如下：

```python
model = MyModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.inference_mode():
        # 在验证集上计算指标
        pass
```

## 21. 学习顺序建议

1. 先掌握张量创建、`shape`、索引、类型和设备。
2. 再掌握形状变换、广播、聚合和矩阵乘法。
3. 理解自动微分、`nn.Module`、损失函数和优化器。
4. 学会用 `Dataset` 和 `DataLoader` 组织数据。
5. 最后进入 CNN、RNN、Attention、混合精度和模型部署。

不需要背 API。记住类别和典型名称，使用时查阅官方文档即可。

## 22. NumPy：数组计算与PyTorch互操作

NumPy 是科学计算的基础库，PyTorch 张量与 NumPy 数组可以共享内存（CPU 张量）。

| 工具 | 用途 |
| --- | --- |
| `np.array(data)` | 从列表或嵌套结构创建数组 |
| `np.zeros/ones/arange/linspace` | 创建常用数组 |
| `arr.reshape` | 改变形状 |
| `np.concatenate/stack` | 拼接数组 |
| `arr.mean/std/sum/max/argmax` | 统计聚合 |
| `torch.from_numpy(arr)` | NumPy → PyTorch（共享内存） |
| `tensor.numpy()` | PyTorch → NumPy（共享内存，CPU 张量） |

In [ ]:
import numpy as np
import torch

# 创建数组
a = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("array:\n", a)
print("zeros:", np.zeros((2, 3)))
print("arange:", np.arange(0, 10, 2))
print("linspace:", np.linspace(0, 1, 5))

# 统计
print("\nmean:", a.mean(), "std:", a.std(), "sum:", a.sum())
print("axis-0 mean:", a.mean(axis=0))
print("argmax:", a.argmax())

# reshape / concatenate
b = a.reshape(3, 2)
print("\nreshaped:", b.shape)
c = np.concatenate([a, a], axis=0)
print("concatenated:", c.shape)

# PyTorch 互操作（共享内存）
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)   # 共享内存
arr[0] = 99.0               # 修改 NumPy 数组
print("\nshared memory demo - tensor:", t)  # 张量也随之改变

t2 = torch.tensor([4.0, 5.0, 6.0])
arr2 = t2.numpy()           # CPU 张量 → NumPy，同样共享内存
print("tensor to numpy:", arr2)

## 23. Pandas：表格数据处理

Pandas 是处理结构化数据（CSV、Excel、数据库）的首选工具，训练前的数据清洗和特征工程通常在这里完成。

| 工具 | 用途 |
| --- | --- |
| `pd.DataFrame(data)` | 从字典或数组创建表格 |
| `pd.read_csv(path)` | 读取 CSV 文件 |
| `df.head/info/describe` | 快速查看数据 |
| `df.dropna/fillna` | 处理缺失值 |
| `df.drop_duplicates` | 去重 |
| `df[col].apply/map` | 逐元素变换 |
| `df.groupby` | 分组聚合 |
| `df.values` / `torch.tensor(df.values)` | 转为 NumPy / PyTorch 张量 |

In [ ]:
import pandas as pd
import torch

# 创建 DataFrame（模拟真实数据集）
data = {
    "age":    [25, 30, None, 22, 35, 28, 30],
    "income": [50000, 60000, 55000, None, 80000, 62000, 60000],
    "edu":    ["bachelor", "master", "bachelor", "bachelor", "phd", "master", "master"],
    "label":  [0, 1, 0, 0, 1, 1, 1],
}
df = pd.DataFrame(data)
print("head:\n", df.head(3))
print("\ninfo:"); df.info()
print("\ndescribe:\n", df.describe())

# 数据清洗
df = df.dropna()                         # 删除含缺失值的行
df = df.drop_duplicates()                # 去重
df["age"] = df["age"].astype(float)

# 特征工程
edu_map = {"bachelor": 0, "master": 1, "phd": 2}
df["edu_code"] = df["edu"].map(edu_map)
df["income_k"] = df["income"].apply(lambda x: x / 1000)  # 单位换算
print("\n处理后:\n", df[["age", "income_k", "edu_code", "label"]])

# groupby 聚合
print("\n按 edu 分组均值:\n", df.groupby("edu")["income"].mean())

# 转为 PyTorch 张量
features = df[["age", "income_k", "edu_code"]].values
labels   = df["label"].values
X = torch.tensor(features, dtype=torch.float32)
y = torch.tensor(labels,   dtype=torch.long)
print("\nfeature tensor:", X.shape, "label tensor:", y.shape)

## 24. Matplotlib：可视化

Matplotlib 是最常用的绘图库，训练中用来观察损失曲线、精度变化、样本图像等。

| 工具 | 用途 |
| --- | --- |
| `plt.plot(x, y)` | 绘制折线图（损失、精度曲线） |
| `plt.scatter(x, y)` | 散点图 |
| `plt.bar(x, height)` | 柱状图 |
| `plt.imshow(image)` | 显示图像或特征图 |
| `plt.subplot(rows, cols, index)` | 多子图布局 |
| `plt.xlabel/ylabel/title/legend` | 标注 |
| `plt.savefig(path)` | 保存图片 |

In [ ]:
import matplotlib
matplotlib.use("Agg")  # 非交互式后端，Notebook 中可去掉这行
import matplotlib.pyplot as plt
import numpy as np

# 模拟训练指标
epochs = list(range(1, 21))
train_loss = [1.0 / (0.3 * e + 1) + np.random.uniform(0, 0.05) for e in epochs]
val_loss   = [1.0 / (0.25 * e + 1) + np.random.uniform(0, 0.08) for e in epochs]
train_acc  = [1 - tl * 0.9 for tl in train_loss]
val_acc    = [1 - vl * 0.9 for vl in val_loss]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 损失曲线
axes[0].plot(epochs, train_loss, label="Train Loss", color="royalblue")
axes[0].plot(epochs, val_loss,   label="Val Loss",   color="tomato", linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss"); axes[0].legend()

# 精度曲线
axes[1].plot(epochs, train_acc, label="Train Acc", color="seagreen")
axes[1].plot(epochs, val_acc,   label="Val Acc",   color="orange", linestyle="--")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training & Validation Accuracy"); axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/training_curves.png", dpi=80)
plt.show()
print("图表已保存至 /tmp/training_curves.png")

## 25. Seaborn：统计可视化

Seaborn 基于 Matplotlib，提供更美观的统计图表，适合探索数据分布和特征相关性。

| 工具 | 用途 |
| --- | --- |
| `sns.histplot(data)` | 直方图，观察分布形状 |
| `sns.kdeplot(data)` | 核密度估计曲线 |
| `sns.heatmap(matrix)` | 热力图（相关矩阵、混淆矩阵） |
| `sns.pairplot(df)` | 多变量两两散点图 |
| `sns.boxplot/violinplot` | 箱线图、提琴图 |

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 模拟特征数据
np.random.seed(0)
df = pd.DataFrame({
    "feature_1": np.random.randn(200),
    "feature_2": np.random.randn(200) * 0.5 + 1,
    "feature_3": np.random.randn(200) * 2 - 1,
})
df["feature_2"] += df["feature_1"] * 0.7  # 制造相关性

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 分布图
sns.histplot(df["feature_1"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Feature 1 Distribution")

# 相关性热力图
corr = df.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Feature Correlation")

plt.tight_layout()
plt.show()